# Grid Overlay Viewer

Interactive notebook for inspecting ISOXML grid cells against the partfield boundary.

This mirrors the logic used by `isoxml-inspect-grid-overlay`, but keeps the result interactive inside Jupyter.

In [6]:
from pathlib import Path

import folium
import geopandas as gpd
from branca.colormap import linear
from IPython.display import display

import isoxml.cli.inspect_grid_overlay as overlay
from isoxml.models import DDEntity


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "pyproject.toml").exists():
        return cwd
    for parent in cwd.parents:
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


REPO_ROOT = resolve_repo_root()
EXAMPLES_DIR = REPO_ROOT / "examples"


def resolve_display_spec(task_data, task, grid, ddi):
    vpn_by_id = {vpn.id: vpn for vpn in getattr(task_data, "value_presentations", []) if vpn.id}
    target_ddi = bytes(ddi)
    zone_candidates = list(task.treatment_zones)
    grid_zone_code = getattr(grid, "treatment_zone_code", None)
    if grid_zone_code is not None:
        zone_candidates.sort(key=lambda zone: 0 if zone.code == grid_zone_code else 1)

    for zone in zone_candidates:
        for pdv in zone.process_data_variables:
            if pdv.process_data_ddi != target_ddi:
                continue
            vpn = vpn_by_id.get(pdv.value_presentation_id_ref)
            if int(grid.type.value) == 2 and vpn is not None:
                scale = float(vpn.scale) / ddi.bit_resolution
                offset = float(vpn.offset)
                unit = vpn.unit_designator or ddi.unit or ddi.name
                return scale, offset, unit
            unit = vpn.unit_designator if vpn and vpn.unit_designator else (ddi.unit or ddi.name)
            return 1.0, 0.0, unit

    return 1.0, 0.0, ddi.unit or ddi.name


In [7]:
SOURCE = EXAMPLES_DIR / "output" / "small_xml_v4_type_2_auto.zip"
TASK_INDEX = 0
GRID_INDEX = 0
DDI = 6
SHOW_ZERO = False
SHP_OVERLAY = None

assert SOURCE.exists(), (
    f"Missing {SOURCE}. Generate one first, for example with "
    "`uv run isoxml-shp-to-prescription ...` or the example bash script."
)
SOURCE


PosixPath('/Users/kunzhou/Documents/codebase/isoxml-py/examples/output/small_xml_v4_type_2_auto.zip')

In [8]:
ddi = DDEntity.from_id(DDI)
task_data, refs = overlay.load(SOURCE)

task = task_data.tasks[TASK_INDEX]
grid = task.grids[GRID_INDEX]
grid_bin = refs.get(grid.filename)
if not isinstance(grid_bin, bytes):
    raise ValueError(f"No binary data found for {grid.filename}.bin")

values = overlay._decode_values(task_data, task, grid, grid_bin, ddi)
display_scale, display_offset, display_unit = resolve_display_spec(task_data, task, grid, ddi)
display_values = values * display_scale + display_offset
cells_gdf = overlay._cells_geodataframe(grid, display_values, SHOW_ZERO)
cells_gdf["value"] = cells_gdf["value"].round(3)
cells_gdf["unit"] = display_unit
partfield_geom = overlay._partfield_geometry(task_data, task)

grid_bbox = overlay.shp.box(
    float(grid.minimum_east_position),
    float(grid.minimum_north_position),
    float(grid.minimum_east_position) + float(grid.cell_east_size) * int(grid.maximum_column),
    float(grid.minimum_north_position) + float(grid.cell_north_size) * int(grid.maximum_row),
)
metrics = overlay._alignment_metrics(grid_bbox, partfield_geom)

summary = {
    "source": str(SOURCE),
    "grid_file": f"{grid.filename}.bin",
    "grid_type": str(grid.type),
    "display_unit": display_unit,
    "rows": int(grid.maximum_row),
    "cols": int(grid.maximum_column),
    "cell_count": len(cells_gdf),
    "min_value": float(display_values.min()),
    "max_value": float(display_values.max()),
    "bbox_overlap": None if metrics is None else round(metrics["overlap"], 6),
    "dx_m": None if metrics is None else round(metrics["dx_m"], 2),
    "dy_m": None if metrics is None else round(metrics["dy_m"], 2),
}
summary


{'source': '/Users/kunzhou/Documents/codebase/isoxml-py/examples/output/small_xml_v4_type_2_auto.zip',
 'grid_file': 'GRD00000.bin',
 'grid_type': 'GridType.GridType2',
 'display_unit': 'kg/ha',
 'rows': 78,
 'cols': 99,
 'cell_count': 4607,
 'min_value': 0.0,
 'max_value': 250.0,
 'bbox_overlap': 0.573285,
 'dx_m': -0.0,
 'dy_m': 0.0}

In [9]:
if cells_gdf.empty:
    raise ValueError("No cells to display. Try setting SHOW_ZERO = True.")

center_geom = partfield_geom if partfield_geom is not None and not partfield_geom.is_empty else cells_gdf.unary_union
center = [center_geom.centroid.y, center_geom.centroid.x]

vmin = float(cells_gdf["value"].min())
vmax = float(cells_gdf["value"].max())
if vmin == vmax:
    vmax = vmin + 1.0
colormap = linear.YlGn_09.scale(vmin, vmax)
colormap.caption = f"Grid value ({display_unit})"

viewer = folium.Map(location=center, zoom_start=16, tiles="OpenStreetMap")

folium.GeoJson(
    cells_gdf.to_json(drop_id=True),
    name="Grid cells",
    style_function=lambda feature: {
        "fillColor": colormap(feature["properties"]["value"]),
        "color": "#3a2618",
        "weight": 0.4,
        "fillOpacity": 0.65,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["row", "col", "value"],
        aliases=["Row", "Column", f"Actual value ({display_unit})"],
        localize=True,
    ),
).add_to(viewer)

if partfield_geom is not None and not partfield_geom.is_empty:
    folium.GeoJson(
        gpd.GeoSeries([partfield_geom], crs="EPSG:4326").to_json(),
        name="Partfield",
        style_function=lambda _: {
            "color": "#d81b60",
            "weight": 3,
            "fillOpacity": 0.0,
            "interactive": False,
        },
    ).add_to(viewer)

if SHP_OVERLAY is not None:
    shp_gdf = gpd.read_file(SHP_OVERLAY).to_crs("EPSG:4326")
    folium.GeoJson(
        shp_gdf.to_json(drop_id=True),
        name="Shapefile overlay",
        style_function=lambda _: {
            "color": "#00acc1",
            "weight": 2,
            "fillOpacity": 0.0,
            "interactive": False,
        },
    ).add_to(viewer)

viewer.fit_bounds([
    [cells_gdf.total_bounds[1], cells_gdf.total_bounds[0]],
    [cells_gdf.total_bounds[3], cells_gdf.total_bounds[2]],
])
colormap.add_to(viewer)
folium.LayerControl().add_to(viewer)
viewer


In [10]:
cells_gdf.head()

,row,col,value,geometry,unit
0,0,59,200.0,"POLYGON ((126.57946 45.57123, 126.57946 45.571...",kg/ha
1,0,60,180.0,"POLYGON ((126.57949 45.57123, 126.57949 45.571...",kg/ha
2,0,61,180.0,"POLYGON ((126.57953 45.57123, 126.57953 45.571...",kg/ha
3,0,62,200.0,"POLYGON ((126.57957 45.57123, 126.57957 45.571...",kg/ha
4,1,56,200.0,"POLYGON ((126.57934 45.57126, 126.57934 45.571...",kg/ha
